# ニャムル音声操作 v0.5.1：データ固定・3条件の比較

現状／二視点dropout／R-Dropを、同じ1,296学習例・68 dev例で比較します。3 seed × 2学習率 × 3条件＝18試行、9モデル。各10 epoch、batch16、float32/SDPA、ModernBERT-ja-30mの同じrevisionです。

1. 「ランタイム」→「ランタイムのタイプを変更」でGPU（T4等）を選ぶ。
2. 「ランタイム」→「すべてのセルを実行」。最初に **nyamuru-voice-v051-inputs.zip** を選ぶ。
3. 以降は環境準備→18試行→devで方法選択を凍結→9モデル評価→検証→保存→ダウンロード。エラーはそのセルで止まります。
4. **nyamuru-voice-v051-colab-results.zip** を保存し、このチャットへ渡す。9モデル分なので前回より大きくなります。ダウンロード完了後にGPUを解放してください。

既存コードを共通利用し、入力は発話本文のみ。追加データ・プロンプト変更なし。学習方法はdevだけで選び、test首位を後から採用しません。

**73件のtestは既に閲覧済みなので回帰確認用です。** 旧版4 splitも回帰確認。独立した精度や新たな95%達成を判定する実験ではありません。改善は未測定で、自動採用はしません。devも小さく、クラス最低2件のため差の解釈に限界があります。

R-Dropの原著は2021年です。2025–2026年のintent研究と比較し、データを固定したまま検証できる方法として採用候補にしました。研究の採否・損失の定義・条件ごとの計算量は同梱の`docs/2026-10-18-v05-research-and-protocol.md`に記載しています。

修正内容：KLをlog-target形式で計算し、係数0ではKLを逆伝播に含めません。停止時は内部ログをセルへ表示し、comparison.logにも保存します。[停止原因と修正記録](docs/2026-10-18-v051-nan-fix.md)。


In [ ]:
from google.colab import files
from pathlib import Path
import hashlib, json, zipfile
uploaded = files.upload()
assert list(uploaded) == ['nyamuru-voice-v051-inputs.zip'], 'Select the Nyamuru input ZIP'
assert hashlib.sha256(uploaded['nyamuru-voice-v051-inputs.zip']).hexdigest() == 'ba1991e526891faf95fa874af2c00c4c6490a99b9b9c41c862e08d865a26f37a', 'Input ZIP differs'
ROOT = Path('/content/nyamuru-voice-v051-20261018')
assert not ROOT.exists(), 'Use a fresh runtime; existing results are not overwritten'
ROOT.mkdir()
with zipfile.ZipFile('nyamuru-voice-v051-inputs.zip') as archive:
    assert all(not Path(n).is_absolute() and '..' not in Path(n).parts for n in archive.namelist())
    archive.extractall(ROOT)
for name, expected in json.loads((ROOT/'input-sha256.json').read_text()).items():
    assert hashlib.sha256((ROOT/name).read_bytes()).hexdigest() == expected, name
del uploaded
print('INPUT HASHES VERIFIED: 19 labels; text only; see data-report.json for counts')


## 環境準備

In [ ]:
from pathlib import Path
import subprocess, sys, time, json
ROOT = Path('/content/nyamuru-voice-v051-20261018')
start = time.perf_counter()
subprocess.run(['nvidia-smi'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
subprocess.run([sys.executable, '-m', 'uv', 'venv', '--python', '3.11.14', str(ROOT/'.venv')], check=True)
PYTHON = str(ROOT/'.venv/bin/python')
subprocess.run([sys.executable, '-m', 'uv', 'pip', 'install', '--python', PYTHON, '-r', str(ROOT/'requirements-train.txt')], check=True)
subprocess.run([PYTHON, '-c', "import torch; assert torch.cuda.is_available(); print(torch.__version__, torch.cuda.get_device_name())"], check=True)
subprocess.run([PYTHON, '-m', 'unittest', 'discover', '-s', str(ROOT), '-p', 'test_*.py', '-v'], cwd=ROOT, check=True)
download = "from huggingface_hub import snapshot_download; snapshot_download('sbintuitions/modernbert-ja-30m', revision='8cb03f54cb9e30e72459e5f1cedc6d89c7d8dcb5', token=False, cache_dir='/content/nyamuru-voice-v051-20261018/hf-cache', allow_patterns=['config.json','model.safetensors','tokenizer.json','tokenizer_config.json','special_tokens_map.json','tokenizer.model','spiece.model','README.md','LICENSE'])"
subprocess.run([PYTHON, '-c', download], check=True)
(ROOT/'setup.json').write_text(json.dumps({'setup_seconds':time.perf_counter()-start, 'notebook_python':sys.version, 'training_python':PYTHON}, indent=2))
(ROOT/'pip-freeze.txt').write_bytes(subprocess.check_output([sys.executable, '-m', 'uv', 'pip', 'freeze', '--python', PYTHON]))
print('SETUP VERIFIED; starting 3 conditions x 6 training trials')


## 学習と結果検証

In [ ]:
from pathlib import Path
import subprocess
ROOT = Path('/content/nyamuru-voice-v051-20261018')
PYTHON = str(ROOT/'.venv/bin/python')
command = [PYTHON, '-u', str(ROOT/'comparison.py'), 'run', '--device', 'cuda']
with (ROOT/'comparison.log').open('x') as log:
    process = subprocess.Popen(command, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        log.write(line); log.flush()
        if '"event"' in line or 'Error' in line or 'Traceback' in line: print(line, end='', flush=True)
    process.stdout.close()
    returncode = process.wait()
if returncode:
    print((ROOT/'comparison.log').read_text()[-16000:])
    raise RuntimeError('Comparison failed; full output saved in comparison.log')
print('18 TRIALS / 9 CHECKPOINTS VERIFIED; method frozen on dev before any regression test')


## 9モデルと比較結果を保存

In [ ]:
from pathlib import Path
import subprocess
ROOT = Path('/content/nyamuru-voice-v051-20261018')
PYTHON = str(ROOT/'.venv/bin/python')
subprocess.run([PYTHON, str(ROOT/'comparison.py'), 'pack', '--output', '/content/nyamuru-voice-v051-colab-results.zip'], cwd=ROOT, check=True)
print((ROOT/'comparison-report.md').read_text())


In [ ]:
from google.colab import files
files.download('/content/nyamuru-voice-v051-colab-results.zip')
